# 04 · Nearest-neighbour search in ten lines
Level L1 · Part 1 · builds on lesson 03

## Goal
By the end of this lesson you can write exact k-nearest-neighbour search over the recipe vectors
with NumPy, as one reusable `knn(query, vectors, k, metric)` function that returns the exact top-k.

## The idea
Lesson 03 ranked five recipes against one query. A search engine asks that question again and
again: "give me the **k** recipes nearest to this query", called **k-nearest-neighbour search**
(k-NN). The simplest answer scores *every* recipe and keeps the best k. This is **brute-force**
or **exact search**: never wrong, but it touches every stored vector on every query.
Think of finding the three closest cafés by measuring the distance to every café in the city:
always correct, and fine for a village, slow for a country.
How fast that cost grows is measured in lesson 27.
Every vector database you meet later is a way to avoid that full scan; this exact answer is their yardstick.

## Picture

```mermaid
flowchart LR
    Q["query [0.3, 0.4]"] --> S["score EVERY recipe<br/>(distance or similarity)"]
    V["recipe matrix<br/>N rows x 2"] --> S
    S --> P["pick the best k<br/>(argsort or argpartition)"]
    P --> R["top-k: names + scores"]
```

The "score every recipe" box costs about N × d arithmetic steps (N recipes, d dimensions). That box is
what grows as the recipe book grows.

## Step by step

### 1. The recipes and the query
Same hand scores as lessons 02 and 03: `[sweet, spicy]`, and the query "mildly spicy and a bit
sweet" is `[0.3, 0.4]`.

In [1]:
import numpy as np

recipe_vectors = {          # [sweet, spicy]
    "Tomato soup":        [0.3, 0.1],
    "Mango salsa":        [0.6, 0.8],
    "Chocolate brownies": [0.95, 0.0],
    "Caesar salad":       [0.1, 0.1],
    "Lemon sorbet":       [0.8, 0.0],
}
names = list(recipe_vectors)
vectors = np.array([recipe_vectors[n] for n in names], dtype=np.float32)
query = [0.3, 0.4]
print("vectors shape:", vectors.shape, " query:", query)

vectors shape: (5, 2)  query: [0.3, 0.4]


### 2. k-NN in ten lines of plain Python
Before NumPy, the whole algorithm with nothing but a loop: measure the distance to every recipe,
sort smallest first, keep the first k. Ten lines including the distance function.

In [2]:
import math

def euclidean(a, b):
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b, strict=True)))

def knn_loop(query, recipe_vectors, k):
    scored = [(euclidean(query, vec), name) for name, vec in recipe_vectors.items()]
    scored.sort()                       # smallest distance first
    return scored[:k]

for dist, name in knn_loop(query, recipe_vectors, k=3):
    print(f"{name:20} distance {dist:.3f}")

Tomato soup          distance 0.300
Caesar salad         distance 0.361
Mango salsa          distance 0.500


Tomato soup (0.300), Caesar salad (0.361) and Mango salsa (0.500): the same top three as
lesson 03's Euclidean ranking. Note the two phases: **score everything**, then **select the top k**.

### 3. One reusable `knn` function with NumPy
Now the same two phases with NumPy, for any of lesson 03's three measures. The function returns
row numbers and scores, so the caller decides what to print. It remembers which direction is
"better": distances smallest first, similarities largest first (we sort `-scores` for those).

In [3]:
def knn(query, vectors, k, metric="euclidean"):
    q = np.asarray(query, dtype=np.float32)
    if metric == "euclidean":
        scores = np.linalg.norm(vectors - q, axis=1)
        order = np.argsort(scores, kind="stable")             # smaller = closer
    elif metric in ("dot", "cosine"):
        scores = vectors @ q
        if metric == "cosine":
            scores = scores / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
        order = np.argsort(-scores, kind="stable")            # bigger = closer
    else:
        raise ValueError(f"unknown metric {metric!r}")
    top = order[:k]
    return top, scores[top]

rows, scores = knn(query, vectors, k=3)
print("rows:  ", rows)
print("scores:", np.round(scores, 3))

rows:   [0 3 1]
scores: [0.3   0.361 0.5  ]


`rows` are positions in `vectors`; `names[i]` turns them back into recipe names.
`kind="stable"` means ties keep their original order, so the result is the same on every run.

### 4. Top-3 with every metric
The same function, three measures. The results match lesson 03 step 6.

In [4]:
def show(rows, scores, label, labels=names, digits=3):
    print(label)
    for rank, (i, s) in enumerate(zip(rows, scores, strict=True), start=1):
        print(f"  {rank}. {labels[i]:20} {s:.{digits}f}")

for metric in ["euclidean", "dot", "cosine"]:
    show(*knn(query, vectors, k=3, metric=metric), label=f"top-3 by {metric}:")

top-3 by euclidean:
  1. Tomato soup          0.300
  2. Caesar salad         0.361
  3. Mango salsa          0.500
top-3 by dot:
  1. Mango salsa          0.500
  2. Chocolate brownies   0.285
  3. Lemon sorbet         0.240
top-3 by cosine:
  1. Mango salsa          1.000
  2. Caesar salad         0.990
  3. Tomato soup          0.822


### 5. Choosing k
**k** is just "how many results do you want". k = 1 answers "what is *the* closest recipe?";
a results page might show 10. Bigger k never changes the order of the first results, it only adds
more below them.

In [5]:
for k in [1, 3, 5]:
    rows, scores = knn(query, vectors, k=k)
    print(f"k={k}:", [names[i] for i in rows])

k=1: ['Tomato soup']
k=3: ['Tomato soup', 'Caesar salad', 'Mango salsa']
k=5: ['Tomato soup', 'Caesar salad', 'Mango salsa', 'Lemon sorbet', 'Chocolate brownies']


Each list starts with the previous one. With k = 5 we get every recipe: k-NN always returns k
results (or all of them, if there are fewer than k), however far away they are.

### 6. "Nearest" is not the same as "good"
Ask for something nothing in the book matches: "very spicy, not sweet at all", `[0.0, 1.0]`.
k-NN still answers.

In [6]:
far_query = [0.0, 1.0]
rows, scores = knn(far_query, vectors, k=3)
show(rows, scores, label="top-3 for 'very spicy, not sweet':")

top-3 for 'very spicy, not sweet':
  1. Mango salsa          0.632
  2. Caesar salad         0.906
  3. Tomato soup          0.949


The best match is Mango salsa at distance 0.632, about twice the 0.300 that Tomato soup scored for our
first query. k-NN ranks; it does not judge. That is why the scores travel with the results:
an application can add a cut-off ("only show results closer than 0.5") on top. Vector databases
offer exactly that as a **score threshold** (lesson 10).

### 7. Grow the recipe book: 100,000 recipes
Five recipes hide the cost. Let's make 100,000 random `[sweet, spicy]` recipes (seeded, so they
are the same every run) and run the same `knn`.

In [7]:
rng = np.random.default_rng(4)
big = rng.random((100_000, 2), dtype=np.float32)        # values in [0, 1)
big_names = [f"recipe {i}" for i in range(len(big))]

rows, scores = knn(query, big, k=3)
for i, s in zip(rows, scores, strict=True):
    print(f"{big_names[i]:13} {np.round(big[i], 3)}  distance {s:.4f}")

recipe 77320  [0.299 0.403]  distance 0.0031
recipe 11485  [0.303 0.399]  distance 0.0034
recipe 3634   [0.303 0.401]  distance 0.0035


With so many recipes, the three nearest sit almost on top of the query (distances around 0.003).
Same code, same exact answer, but now every query computed 100,000 distances.
`knn` also sorts all 100,000 distances just to keep 3. Step 8 fixes that.

### 8. Select the top k without sorting everything: `np.argpartition`
`np.argpartition(scores, k - 1)` moves the k smallest scores to the front **without sorting the
rest**. NumPy only promises *which* items are in the first k positions, not their order, so we
then sort just those k. For similarities, partition `-scores`, as in step 3.

In [8]:
def top_k_smallest(scores, k):
    k = min(k, len(scores))
    part = np.argpartition(scores, k - 1)[:k]           # the k best, in no promised order
    return part[np.argsort(scores[part], kind="stable")]  # sort only those k

import time

def avg_ms(fn, repeats=10):                             # average wall time of fn(), in ms
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) / repeats * 1000

q = np.array(query, dtype=np.float32)
data = rng.random((1_000_000, 2), dtype=np.float32)
dists = np.linalg.norm(data - q, axis=1)
same = np.array_equal(top_k_smallest(dists, 3), np.argsort(dists, kind="stable")[:3])
print("same top-3 as a full argsort:", same)
print(f"full argsort:      {avg_ms(lambda: np.argsort(dists)[:3]):6.2f} ms")
print(f"argpartition + k:  {avg_ms(lambda: top_k_smallest(dists, 3)):6.2f} ms")

same top-3 as a full argsort: True


full argsort:       62.09 ms
argpartition + k:    5.47 ms


Same answer, and on a million scores the partition is several times faster than the full sort
(the exact ratio depends on your machine). Selecting is now cheap; what remains is the scoring,
which exact search cannot skip. Production libraries do exactly this: score everything, then a
partial selection (FAISS calls its exact index `IndexFlat`; Qdrant runs it when you ask for
`exact=True`).

### 9. The final `knn`
Put the faster selection into the function. This is the brute-force baseline the rest of the
course compares against: "how many of the exact top-k did an index find?" (recall, lesson 16)
is measured against this answer, starting with lesson 28's IVF index.

In [9]:
def knn(query, vectors, k, metric="euclidean"):
    q = np.asarray(query, dtype=np.float32)
    if metric == "euclidean":
        scores = np.linalg.norm(vectors - q, axis=1)
        rows = top_k_smallest(scores, k)                      # smaller = closer
    elif metric in ("dot", "cosine"):
        scores = vectors @ q
        if metric == "cosine":
            scores = scores / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
        rows = top_k_smallest(-scores, k)                     # bigger = closer
    else:
        raise ValueError(f"unknown metric {metric!r}")
    return rows, scores[rows]

show(*knn(query, vectors, k=3, metric="cosine"), label="toy book, cosine:")
show(*knn(query, big, k=3), label="100,000 recipes, euclidean:", labels=big_names, digits=4)

toy book, cosine:
  1. Mango salsa          1.000
  2. Caesar salad         0.990
  3. Tomato soup          0.822
100,000 recipes, euclidean:
  1. recipe 77320         0.0031
  2. recipe 11485         0.0034
  3. recipe 3634          0.0035


The toy result matches step 4's cosine list, and the 100,000-recipe result matches step 7.

## What just happened
- Ten lines of plain Python scored all five recipes and kept the best three (step 2): Tomato soup, Caesar salad, Mango salsa.
- One NumPy `knn` function did the same for Euclidean, dot and cosine, sorting each in its own direction (steps 3–4).
- k sets how many results come back; k-NN returns them even when the best is a poor 0.632 (steps 5–6).
- On 100,000 recipes the same `knn` still gave the exact answer, at 100,000 distances per query (step 7).
- `np.argpartition` picked the same top-3 from a million scores without sorting everything, several times faster than a full sort (step 8); the final `knn` is the exact baseline for later lessons (step 9).

## Common mistakes

**1. Asking for more neighbours than there are recipes.** Slicing `argsort(...)[:10]` quietly
returns all 5 rows, but `np.argpartition` needs a position that exists:

In [10]:
try:
    dists_toy = np.linalg.norm(vectors - q, axis=1)
    k = 10
    np.argpartition(dists_toy, k - 1)[:k]
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: kth(=9) out of bounds (5)


It prints `ValueError: kth(=9) out of bounds (5)`: there is no 10th recipe to partition around.
Fix: cap k at the number of vectors, which is the first line of `top_k_smallest`.

In [11]:
rows = top_k_smallest(dists_toy, 10)
print(f"asked for 10, got {len(rows)}:", [names[i] for i in rows])

asked for 10, got 5: ['Tomato soup', 'Caesar salad', 'Mango salsa', 'Lemon sorbet', 'Chocolate brownies']


**2. Trusting the order of `argpartition`.** It is tempting to drop the second sort and use the
partition directly. With k = 3 that often *happens* to come out in order, which hides the bug.
Ask for more results, say 1,000 candidates from the million recipes, and check the order:

In [12]:
try:
    k = 1_000
    rows = np.argpartition(dists, k - 1)[:k]                # no second sort
    assert np.all(np.diff(dists[rows]) >= 0), "top-k rows are not sorted by distance"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AssertionError: top-k rows are not sorted by distance


It prints `AssertionError: top-k rows are not sorted by distance`: the right 1,000 recipes, in a
jumbled order. Fix: always sort the k selected rows by their scores, as `top_k_smallest` does.

In [13]:
rows = top_k_smallest(dists, 1_000)
print("sorted after the second step:", bool(np.all(np.diff(dists[rows]) >= 0)))

sorted after the second step: True


**3. "More like this" returns the recipe itself.** A common feature is "show recipes like Mango
salsa": use Mango salsa's own vector as the query. But Mango salsa is in the book, at distance 0,
so it always comes first:

In [14]:
try:
    salsa = names.index("Mango salsa")
    rows, scores = knn(vectors[salsa], vectors, k=2)
    print([names[i] for i in rows], np.round(scores, 3))
    assert salsa not in rows, "the recommendations include Mango salsa itself"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

['Mango salsa', 'Tomato soup'] [0.    0.762]
AssertionError: the recommendations include Mango salsa itself


The list starts with Mango salsa at 0.000, and the check prints
`AssertionError: the recommendations include Mango salsa itself`. Fix: ask for one extra
neighbour and drop the query's own row.

In [15]:
rows, scores = knn(vectors[salsa], vectors, k=2 + 1)
keep = rows != salsa
print("like Mango salsa:", [names[i] for i in rows[keep]], np.round(scores[keep], 3))

like Mango salsa: ['Tomato soup', 'Lemon sorbet'] [0.762 0.825]


## Try it
1. **Sweet tooth.** Use `knn` to find the top-2 recipes for "very sweet, not spicy", `[1.0, 0.0]`,
   with Euclidean and with cosine. Do they agree? Then run the same query on the 100,000-recipe
   `big` matrix with k = 5 and print the distances.
2. **Many queries at once.** Write `knn_batch(queries, vectors, k)` that takes a `(m, 2)` matrix of
   queries and returns an `(m, k)` array of row numbers using the dot product, with one matrix
   multiplication (`queries @ vectors.T`) and `np.argpartition(..., axis=1)`. Check it against
   calling `knn(..., metric="dot")` once per query.
   Hint: `np.take_along_axis(scores, part, axis=1)` picks each row's selected scores.

In [16]:
# your code here

## Key terms
- **k-nearest neighbours (k-NN)** — the k stored vectors closest to a query under a chosen metric.
- **top-k** — the k best-scoring results, in order; k is chosen by the caller.
- **exact search (brute force)** — scoring every stored vector for every query; always finds the true top-k, costs N × d work per query.
- **argpartition** — NumPy's partial selection: moves the k smallest values to the front without sorting the rest (and without ordering those k).